# 05 Model Comparison

fixed A/B の予測評価結果をモデル横断で統合する。ここでは新しいモデル推定は行わず、`output/forecasts/metrics/` に保存済みの metrics を読み込んで、同じ表と図で比較する。

比較対象は naive / seasonal naive / SARIMA fixed / SARIMAX fixed / SARIMA grid best / SARIMAX grid best / SSM / Prophet である。SARIMAX と SSM は test 期間の外生ダミーを既知として与えた conditional forecast なので、unconditional forecast との単純比較には注意する。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "Transport_amount_project" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

FORECAST_DIR = PROJECT_ROOT / "output" / "forecasts"
METRICS_DIR = FORECAST_DIR / "metrics"
FIGURES_DIR = FORECAST_DIR / "figures"

METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Metrics dir:", METRICS_DIR)
print("Figures dir:", FIGURES_DIR)

## 1. Metrics ファイルの読み込み

候補ファイルが存在しない場合は notebook を止めず、skip 理由を `load_log` に残す。モデル名は比較用に整理し、情報条件を `unconditional` と `conditional_exog_known` に分ける。

In [ ]:
metric_files = {
    "naive": METRICS_DIR / "naive_metrics.csv",
    "sarimax_fixed": METRICS_DIR / "sarimax_metrics.csv",
    "sarimax_grid_best": METRICS_DIR / "sarimax_grid_best_models.csv",
    "ssm": METRICS_DIR / "ssm_metrics.csv",
    "prophet": METRICS_DIR / "prophet_metrics.csv",
}

load_log = []


def require_columns(df: pd.DataFrame, columns: list[str], source_name: str) -> None:
    missing = [col for col in columns if col not in df.columns]
    if missing:
        raise ValueError(f"{source_name} is missing columns: {missing}")


def read_metrics(source_name: str) -> pd.DataFrame | None:
    path = metric_files[source_name]
    if not path.exists():
        load_log.append({"source": source_name, "path": str(path), "status": "skipped", "reason": "file not found"})
        return None
    try:
        df = pd.read_csv(path)
        load_log.append({"source": source_name, "path": str(path), "status": "loaded", "reason": ""})
        return df
    except Exception as exc:
        load_log.append({"source": source_name, "path": str(path), "status": "skipped", "reason": str(exc)})
        return None


rows = []

naive = read_metrics("naive")
if naive is not None:
    require_columns(naive, ["model", "split", "forecast_type", "rmse", "mae", "mape", "mase"], "naive_metrics")
    for _, row in naive.iterrows():
        rows.append(
            {
                "split": row["split"],
                "model": row["model"],
                "forecast_type": "unconditional",
                "rmse": row["rmse"],
                "mae": row["mae"],
                "mape": row["mape"],
                "mase": row["mase"],
                "notes": "recursive fixed-split baseline; seasonal_naive uses generated test-period forecasts for horizons beyond 12 months",
            }
        )

sarimax_fixed = read_metrics("sarimax_fixed")
if sarimax_fixed is not None:
    require_columns(sarimax_fixed, ["model", "split", "forecast_type", "rmse", "mae", "mape", "mase"], "sarimax_metrics")
    fixed_name_map = {"sarima": "sarima_fixed", "sarimax": "sarimax_fixed"}
    for _, row in sarimax_fixed.iterrows():
        model_name = fixed_name_map.get(row["model"], row["model"])
        notes = "fixed-order SARIMA baseline"
        if model_name == "sarimax_fixed":
            notes = "fixed-order SARIMAX; conditional forecast with test-period exogenous dummies known"
        rows.append(
            {
                "split": row["split"],
                "model": model_name,
                "forecast_type": row["forecast_type"],
                "rmse": row["rmse"],
                "mae": row["mae"],
                "mape": row["mape"],
                "mase": row["mase"],
                "notes": notes,
            }
        )

grid_best = read_metrics("sarimax_grid_best")
if grid_best is not None:
    require_columns(grid_best, ["model", "split", "rmse", "mae", "mape", "mase"], "sarimax_grid_best_models")
    grid_name_map = {"sarima": "sarima_grid_best", "sarimax": "sarimax_grid_best"}
    for _, row in grid_best.iterrows():
        model_name = grid_name_map.get(row["model"], row["model"])
        forecast_type = "conditional" if model_name == "sarimax_grid_best" else "unconditional"
        notes = "small SARIMA order grid best by converged RMSE"
        if model_name == "sarimax_grid_best":
            notes = "small SARIMAX order grid best by converged RMSE; conditional forecast with test-period exogenous dummies known"
        rows.append(
            {
                "split": row["split"],
                "model": model_name,
                "forecast_type": forecast_type,
                "rmse": row["rmse"],
                "mae": row["mae"],
                "mape": row["mape"],
                "mase": row["mase"],
                "notes": notes,
            }
        )

ssm = read_metrics("ssm")
if ssm is not None:
    require_columns(ssm, ["split", "forecast_type", "rmse", "mae", "mape", "mase"], "ssm_metrics")
    for _, row in ssm.iterrows():
        rows.append(
            {
                "split": row["split"],
                "model": "ssm_conditional",
                "forecast_type": "conditional",
                "rmse": row["rmse"],
                "mae": row["mae"],
                "mape": row["mape"],
                "mase": row["mase"],
                "notes": "main-analysis SSM; conditional forecast with test-period exogenous dummies known",
            }
        )

prophet = read_metrics("prophet")
if prophet is not None:
    require_columns(prophet, ["split", "forecast_type", "rmse", "mae", "mape", "mase"], "prophet_metrics")
    for _, row in prophet.iterrows():
        rows.append(
            {
                "split": row["split"],
                "model": "prophet",
                "forecast_type": "unconditional",
                "rmse": row["rmse"],
                "mae": row["mae"],
                "mape": row["mape"],
                "mase": row["mase"],
                "notes": "Prophet on log scale; no holidays and no regressors",
            }
        )

load_log_df = pd.DataFrame(load_log)
comparison = pd.DataFrame(rows)
if comparison.empty:
    raise RuntimeError("No model metrics were loaded.")

comparison["information_set"] = comparison["forecast_type"].map(
    {"conditional": "conditional_exog_known", "unconditional": "unconditional"}
).fillna(comparison["forecast_type"])

display(load_log_df)
display(comparison.sort_values(["split", "rmse"]))

## 2. 統合表と順位

split 内で RMSE / MAPE / MASE の順位を付ける。順位は値が小さいほど良い。

In [ ]:
comparison = comparison.copy()
for metric in ["rmse", "mape", "mase"]:
    comparison[f"{metric}_rank_within_split"] = comparison.groupby("split")[metric].rank(method="min")

comparison = comparison[
    [
        "split",
        "model",
        "forecast_type",
        "information_set",
        "rmse",
        "mae",
        "mape",
        "mase",
        "rmse_rank_within_split",
        "mape_rank_within_split",
        "mase_rank_within_split",
        "notes",
    ]
]

best_by_split = (
    comparison.sort_values(["split", "rmse"])
    .groupby("split", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

best_unconditional = (
    comparison[comparison["information_set"] == "unconditional"]
    .sort_values(["split", "rmse"])
    .groupby("split", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

display(comparison.sort_values(["split", "rmse"]))
display(best_by_split)
display(best_unconditional)

## 3. 保存

統合表と split 別 best model 表を `output/forecasts/metrics/` に保存する。

In [ ]:
comparison_path = METRICS_DIR / "model_comparison_fixed_splits.csv"
best_path = METRICS_DIR / "model_comparison_best_by_split.csv"

comparison.sort_values(["split", "rmse"]).to_csv(comparison_path, index=False)
best_by_split.to_csv(best_path, index=False)

print("Saved:", comparison_path)
print("Saved:", best_path)

## 4. 図の作成

fixed A/B を分け、モデル別に RMSE と MAPE を比較する。バーの色で unconditional と conditional forecast を区別する。

In [ ]:
info_colors = {
    "unconditional": "#4C78A8",
    "conditional_exog_known": "#F58518",
}


def plot_metric(metric: str, ylabel: str, save_path: Path) -> None:
    split_names = sorted(comparison["split"].unique())
    fig, axes = plt.subplots(nrows=1, ncols=len(split_names), figsize=(13, 5), sharey=False)
    if len(split_names) == 1:
        axes = [axes]

    for ax, split_name in zip(axes, split_names):
        plot_df = comparison[comparison["split"] == split_name].sort_values(metric).copy()
        labels = [f"{row.model}\n{'C' if row.information_set == 'conditional_exog_known' else 'U'}" for row in plot_df.itertuples()]
        colors = [info_colors[value] for value in plot_df["information_set"]]
        ax.bar(labels, plot_df[metric], color=colors)
        ax.set_title(split_name)
        ax.set_ylabel(ylabel)
        ax.tick_params(axis="x", labelrotation=45)
        ax.grid(True, axis="y", color="0.85", linewidth=0.8)

    handles = [
        plt.Rectangle((0, 0), 1, 1, color=color, label=label)
        for label, color in info_colors.items()
    ]
    fig.legend(handles=handles, loc="upper center", ncol=2, frameon=False)
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


plot_metric("rmse", "RMSE", FIGURES_DIR / "model_comparison_rmse.png")
plot_metric("mape", "MAPE", FIGURES_DIR / "model_comparison_mape.png")

print("Saved model comparison figures.")

## 5. 解釈メモ

- fixed A では Prophet が最も低い RMSE で、unconditional モデルとしても全体としても有利である。SARIMAX grid best と SSM conditional も良いが、外生ダミー既知の conditional forecast である点に注意する。
- fixed B では SARIMAX grid best が最も低い RMSE で、SSM conditional もほぼ同水準に強い。これは test 期間の外生ダミーを既知として与えている conditional forecast の結果である。
- unconditional 同士では、fixed A は Prophet、fixed B は seasonal naive が最も良い。Prophet は fixed A では良いが、fixed B では seasonal naive や SARIMA より弱い。
- conditional を含めると、fixed A は Prophet、fixed B は SARIMAX grid best が best になる。
- SARIMAX/SSM は conditional forecast なので、Prophet/SARIMA/seasonal naive と単純に横並びで優劣を決めるのではなく、情報条件の違いを明示して読む必要がある。
- Autoformer を追加したら、この比較表に同じ共通フォーマットの metrics を追加して比較する予定である。